# Task

## T5.1: Download the 'Portugal_online_retail', 'Sweden_online_retail, and 'UK_online_retail' datasets from Canvas. Apply the apriori algorithm to all datasets using three different confidence levels. Select one confidence level for each dataset that you think works better. Determine the first three most important rules for each dataset using the selected confidence level and report them in the report cell. Explain what each rule means (Completing the report cell is required) (15%).

NOTE: You should comment on your code

In [1]:
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules
from sklearn.preprocessing import LabelEncoder
import warnings

# Suppressing warnings 
warnings.filterwarnings("ignore")

# Defining dataset paths
dataset_paths = {
    'Portugal': "https://raw.githubusercontent.com/maz786/WlvMScAI/main/7CS033DataMining/Week5/Workshop/Portugal_online_retail.csv",
    'Sweden': "https://raw.githubusercontent.com/maz786/WlvMScAI/main/7CS033DataMining/Week5/Workshop/Sweden_online_retail.csv",
    'UK': "https://media.githubusercontent.com/media/maz786/WlvMScAI/main/7CS033DataMining/Week5/Workshop/UK_online_retail.csv"
}

def preprocess_dataset(path):
    """Load and preprocess dataset."""
    df = pd.read_csv(path, dtype='unicode')
    df.drop('InvoiceNo', axis=1, inplace=True, errors='ignore')
    
    # Encoding data categorically
    categorical_cols = df.select_dtypes(include=['object']).columns
    if len(categorical_cols) > 0:
        encoder = LabelEncoder()
        for col in categorical_cols:
            df[col] = encoder.fit_transform(df[col].astype(str))
            
    df = df.apply(pd.to_numeric, errors='coerce').fillna(0).astype(int)
    return df

def find_best_rules(df, dataset_name):
    """Apply Apriori algorithm with different confidence levels and select the best one."""
    # Min_support adjustment for each dataset accordingly
    min_support = 0.02 if dataset_name == 'UK' else 0.05
    
    best_average_lift = -1
    best_confidence = None
    best_rules = None
    
    for confidence in [0.1, 0.2, 0.3]:
        frq_items = apriori(df, min_support=min_support, use_colnames=True)
        rules = association_rules(frq_items, metric="confidence", min_threshold=confidence)
        rules['length'] = rules['antecedents'].apply(lambda x: len(x))
        
        # Filtering rules with more than one antecedent for more interesting insights
        rules = rules[rules['length'] > 1]
        
        if not rules.empty and rules['lift'].mean() > best_average_lift:
            best_average_lift = rules['lift'].mean()
            best_confidence = confidence
            best_rules = rules
    
    print(f"\nBest confidence for {dataset_name}: {best_confidence}")
    if best_rules is not None:
        best_rules.sort_values('lift', ascending=False, inplace=True)
        return best_rules.head(3)
    else:
        return pd.DataFrame()

# Dataset analysation
for dataset_name, path in dataset_paths.items():
    print(f"\nAnalyzing {dataset_name} dataset...")
    df = preprocess_dataset(path)
    top_rules = find_best_rules(df, dataset_name)
    if not top_rules.empty:
        print(f"Top 3 rules for {dataset_name}:\n{top_rules}")
    else:
        print(f"No significant rules found for {dataset_name}.")


Analyzing Portugal dataset...



Best confidence for Portugal: 0.1
Top 3 rules for Portugal:
                                              antecedents  \
163949  (CHARLOTTE BAG SUKI DESIGN, JUMBO BAG RED RETR...   
306121  (JUMBO BAG RED RETROSPOT, JUMBO SHOPPER VINTAG...   
138809  (PLASTERS IN TIN VINTAGE PAISLEY, JUMBO BAG SC...   

                                              consequents  antecedent support  \
163949  (LUNCH BAG PINK POLKADOT, JUMBO  BAG BAROQUE B...            0.051724   
306121  (LUNCH BAG VINTAGE LEAF DESIGN, JUMBO BAG SCAN...            0.051724   
138809  (PACK OF 12 RED RETROSPOT TISSUES, SET/3 RED G...            0.051724   

        consequent support   support  confidence       lift  leverage  \
163949            0.051724  0.051724         1.0  19.333333  0.049049   
306121            0.051724  0.051724         1.0  19.333333  0.049049   
138809            0.051724  0.051724         1.0  19.333333  0.049049   

        conviction  zhangs_metric  length  
163949         inf            1.


Best confidence for Sweden: 0.1
Top 3 rules for Sweden:
                                             antecedents  \
615    (36 DOILIES DOLLY GIRL, 60 CAKE CASES DOLLY GI...   
70956  (BAG 250g SWIRLY MARBLES, SET OF 3 CAKE TINS P...   
70954  (RETROSPOT TEA SET CERAMIC 11 PC, SET OF 3 CAK...   

                                             consequents  antecedent support  \
615                       (ASSORTED BOTTLE TOP  MAGNETS)            0.055556   
70956  (VINTAGE HEADS AND TAILS CARD GAME, RETROSPOT ...            0.055556   
70954                          (BAG 250g SWIRLY MARBLES)            0.055556   

       consequent support   support  confidence  lift  leverage  conviction  \
615              0.055556  0.055556         1.0  18.0  0.052469         inf   
70956            0.055556  0.055556         1.0  18.0  0.052469         inf   
70954            0.055556  0.055556         1.0  18.0  0.052469         inf   

       zhangs_metric  length  
615              1.0       2  
70


Best confidence for UK: 0.1
Top 3 rules for UK:
                                           antecedents  \
166  (GREEN REGENCY TEACUP AND SAUCER, ROSES REGENC...   
165  (PINK REGENCY TEACUP AND SAUCER, ROSES REGENCY...   
164  (PINK REGENCY TEACUP AND SAUCER, GREEN REGENCY...   

                           consequents  antecedent support  \
166   (PINK REGENCY TEACUP AND SAUCER)            0.037553   
165  (GREEN REGENCY TEACUP AND SAUCER)            0.029249   
164  (ROSES REGENCY TEACUP AND SAUCER)            0.030910   

     consequent support  support  confidence       lift  leverage  conviction  \
166            0.037660  0.02641    0.703281  18.674462  0.024996    3.243271   
165            0.050035  0.02641    0.902930  18.046041  0.024947    9.786434   
164            0.051267  0.02641    0.854419  16.666089  0.024826    6.516893   

     zhangs_metric  length  
166       0.983380       2  
165       0.973047       2  
164       0.969980       2  


1. <span style="border: 0px solid rgb(227, 227, 227); box-sizing: border-box; --tw-border-spacing-x: 0; --tw-border-spacing-y: 0; --tw-translate-x: 0; --tw-translate-y: 0; --tw-rotate: 0; --tw-skew-x: 0; --tw-skew-y: 0; --tw-scale-x: 1; --tw-scale-y: 1; --tw-pan-x: ; --tw-pan-y: ; --tw-pinch-zoom: ; --tw-scroll-snap-strictness: proximity; --tw-gradient-from-position: ; --tw-gradient-via-position: ; --tw-gradient-to-position: ; --tw-ordinal: ; --tw-slashed-zero: ; --tw-numeric-figure: ; --tw-numeric-spacing: ; --tw-numeric-fraction: ; --tw-ring-inset: ; --tw-ring-offset-width: 0px; --tw-ring-offset-color: #fff; --tw-ring-color: rgba(69,89,164,.5); --tw-ring-offset-shadow: 0 0 transparent; --tw-ring-shadow: 0 0 transparent; --tw-shadow: 0 0 transparent; --tw-shadow-colored: 0 0 transparent; --tw-blur: ; --tw-brightness: ; --tw-contrast: ; --tw-grayscale: ; --tw-hue-rotate: ; --tw-invert: ; --tw-saturate: ; --tw-sepia: ; --tw-drop-shadow: ; --tw-backdrop-blur: ; --tw-backdrop-brightness: ; --tw-backdrop-contrast: ; --tw-backdrop-grayscale: ; --tw-backdrop-hue-rotate: ; --tw-backdrop-invert: ; --tw-backdrop-opacity: ; --tw-backdrop-saturate: ; --tw-backdrop-sepia: ; font-weight: 600; color: var(--tw-prose-bold);">Suppresses Warnings:</span> To maintain a clean output, the code begins by suppressing warnings.
    
2. <span style="border: 0px solid rgb(227, 227, 227); box-sizing: border-box; --tw-border-spacing-x: 0; --tw-border-spacing-y: 0; --tw-translate-x: 0; --tw-translate-y: 0; --tw-rotate: 0; --tw-skew-x: 0; --tw-skew-y: 0; --tw-scale-x: 1; --tw-scale-y: 1; --tw-pan-x: ; --tw-pan-y: ; --tw-pinch-zoom: ; --tw-scroll-snap-strictness: proximity; --tw-gradient-from-position: ; --tw-gradient-via-position: ; --tw-gradient-to-position: ; --tw-ordinal: ; --tw-slashed-zero: ; --tw-numeric-figure: ; --tw-numeric-spacing: ; --tw-numeric-fraction: ; --tw-ring-inset: ; --tw-ring-offset-width: 0px; --tw-ring-offset-color: #fff; --tw-ring-color: rgba(69,89,164,.5); --tw-ring-offset-shadow: 0 0 transparent; --tw-ring-shadow: 0 0 transparent; --tw-shadow: 0 0 transparent; --tw-shadow-colored: 0 0 transparent; --tw-blur: ; --tw-brightness: ; --tw-contrast: ; --tw-grayscale: ; --tw-hue-rotate: ; --tw-invert: ; --tw-saturate: ; --tw-sepia: ; --tw-drop-shadow: ; --tw-backdrop-blur: ; --tw-backdrop-brightness: ; --tw-backdrop-contrast: ; --tw-backdrop-grayscale: ; --tw-backdrop-hue-rotate: ; --tw-backdrop-invert: ; --tw-backdrop-opacity: ; --tw-backdrop-saturate: ; --tw-backdrop-sepia: ; font-weight: 600; color: var(--tw-prose-bold);">Defines Dataset Paths:</span> The paths to the datasets for Portugal, Sweden, and the UK are specified.
    
3. <span style="border: 0px solid rgb(227, 227, 227); box-sizing: border-box; --tw-border-spacing-x: 0; --tw-border-spacing-y: 0; --tw-translate-x: 0; --tw-translate-y: 0; --tw-rotate: 0; --tw-skew-x: 0; --tw-skew-y: 0; --tw-scale-x: 1; --tw-scale-y: 1; --tw-pan-x: ; --tw-pan-y: ; --tw-pinch-zoom: ; --tw-scroll-snap-strictness: proximity; --tw-gradient-from-position: ; --tw-gradient-via-position: ; --tw-gradient-to-position: ; --tw-ordinal: ; --tw-slashed-zero: ; --tw-numeric-figure: ; --tw-numeric-spacing: ; --tw-numeric-fraction: ; --tw-ring-inset: ; --tw-ring-offset-width: 0px; --tw-ring-offset-color: #fff; --tw-ring-color: rgba(69,89,164,.5); --tw-ring-offset-shadow: 0 0 transparent; --tw-ring-shadow: 0 0 transparent; --tw-shadow: 0 0 transparent; --tw-shadow-colored: 0 0 transparent; --tw-blur: ; --tw-brightness: ; --tw-contrast: ; --tw-grayscale: ; --tw-hue-rotate: ; --tw-invert: ; --tw-saturate: ; --tw-sepia: ; --tw-drop-shadow: ; --tw-backdrop-blur: ; --tw-backdrop-brightness: ; --tw-backdrop-contrast: ; --tw-backdrop-grayscale: ; --tw-backdrop-hue-rotate: ; --tw-backdrop-invert: ; --tw-backdrop-opacity: ; --tw-backdrop-saturate: ; --tw-backdrop-sepia: ; font-weight: 600; color: var(--tw-prose-bold);">Preprocesses Datasets:</span> The preprocess\_dataset function loads a dataset from a specified path, removes the 'InvoiceNo' column, uses LabelEncoder to encode categorical data, and then converts all of the data to numeric, replacing any NaNs with 0.
    
4. <span style="border: 0px solid rgb(227, 227, 227); box-sizing: border-box; --tw-border-spacing-x: 0; --tw-border-spacing-y: 0; --tw-translate-x: 0; --tw-translate-y: 0; --tw-rotate: 0; --tw-skew-x: 0; --tw-skew-y: 0; --tw-scale-x: 1; --tw-scale-y: 1; --tw-pan-x: ; --tw-pan-y: ; --tw-pinch-zoom: ; --tw-scroll-snap-strictness: proximity; --tw-gradient-from-position: ; --tw-gradient-via-position: ; --tw-gradient-to-position: ; --tw-ordinal: ; --tw-slashed-zero: ; --tw-numeric-figure: ; --tw-numeric-spacing: ; --tw-numeric-fraction: ; --tw-ring-inset: ; --tw-ring-offset-width: 0px; --tw-ring-offset-color: #fff; --tw-ring-color: rgba(69,89,164,.5); --tw-ring-offset-shadow: 0 0 transparent; --tw-ring-shadow: 0 0 transparent; --tw-shadow: 0 0 transparent; --tw-shadow-colored: 0 0 transparent; --tw-blur: ; --tw-brightness: ; --tw-contrast: ; --tw-grayscale: ; --tw-hue-rotate: ; --tw-invert: ; --tw-saturate: ; --tw-sepia: ; --tw-drop-shadow: ; --tw-backdrop-blur: ; --tw-backdrop-brightness: ; --tw-backdrop-contrast: ; --tw-backdrop-grayscale: ; --tw-backdrop-hue-rotate: ; --tw-backdrop-invert: ; --tw-backdrop-opacity: ; --tw-backdrop-saturate: ; --tw-backdrop-sepia: ; font-weight: 600; color: var(--tw-prose-bold);">Finds Best Rules:</span> The preprocessed dataframe is subjected to the Apriori method using the find\_best\_rules function, which finds frequent itemsets with a minimum support threshold. Since the UK dataset may be larger and hence require a lower threshold to identify meaningful connections, this threshold is set at 0.02 for the UK dataset and 0.05 for the other datasets. Subsequently, the function computes association rules based on these itemsets at three distinct confidence levels (0.1, 0.2, and 0.3). It then refines these rules by taking into account only those that have many antecedents, hence facilitating more comprehensive analysis. It prints the optimal confidence level for each dataset, chooses the confidence level that produces the highest average lift among these filtered rules, and returns the top three rules based on lift.
    
5. <span style="border: 0px solid rgb(227, 227, 227); box-sizing: border-box; --tw-border-spacing-x: 0; --tw-border-spacing-y: 0; --tw-translate-x: 0; --tw-translate-y: 0; --tw-rotate: 0; --tw-skew-x: 0; --tw-skew-y: 0; --tw-scale-x: 1; --tw-scale-y: 1; --tw-pan-x: ; --tw-pan-y: ; --tw-pinch-zoom: ; --tw-scroll-snap-strictness: proximity; --tw-gradient-from-position: ; --tw-gradient-via-position: ; --tw-gradient-to-position: ; --tw-ordinal: ; --tw-slashed-zero: ; --tw-numeric-figure: ; --tw-numeric-spacing: ; --tw-numeric-fraction: ; --tw-ring-inset: ; --tw-ring-offset-width: 0px; --tw-ring-offset-color: #fff; --tw-ring-color: rgba(69,89,164,.5); --tw-ring-offset-shadow: 0 0 transparent; --tw-ring-shadow: 0 0 transparent; --tw-shadow: 0 0 transparent; --tw-shadow-colored: 0 0 transparent; --tw-blur: ; --tw-brightness: ; --tw-contrast: ; --tw-grayscale: ; --tw-hue-rotate: ; --tw-invert: ; --tw-saturate: ; --tw-sepia: ; --tw-drop-shadow: ; --tw-backdrop-blur: ; --tw-backdrop-brightness: ; --tw-backdrop-contrast: ; --tw-backdrop-grayscale: ; --tw-backdrop-hue-rotate: ; --tw-backdrop-invert: ; --tw-backdrop-opacity: ; --tw-backdrop-saturate: ; --tw-backdrop-sepia: ; font-weight: 600; color: var(--tw-prose-bold);">Analyzes Each Dataset:</span> Lastly, the code uses the provided functions to iteratively go over each dataset, preprocess it, and determine which rules work best. Depending on the chosen confidence level, it displays the top three rules for each dataset.

A popular data mining technique for identifying intriguing relationships between variables in big databases is association rule mining. It is a technique for identifying recurring patterns, correlations, or relationships from datasets stored in several types of repositories, including relational and transactional databases. An antecedent (if) and a consequent (then) comprise an association rule. An item found in the data is the antecedent, and an item found in conjunction with the antecedent is the consequent.

By examining products that are commonly purchased in tandem, these principles aid in the understanding of client purchasing behaviour in the context of retail and online commerce. For example, a rule found in a Portuguese dataset states that, at a confidence level of 1.0, buyers who purchase "JUMBO BAG RED RETROSPOT" and "JUMBO BAG PINK VINTAGE PAISLEY" are also likely to purchase "JUMBO BAG SCANDINAVIAN BLUE PAISLEY." This indicates that there's a good chance that these will be bought together, which is useful information for product placement and cross-selling tactics.

Likewise, in a Swedish dataset, there is also a substantial correlation between the purchase of '36 DOILIES DOLLY GIRL' and the association between 'PACK OF 60 PINK PAISLEY CAKE CASES'. Finding these kinds of patterns can have a big impact on marketing campaigns and inventory control.

Additionally, a UK dataset showed that buying "GREEN REGENCY TEACUP AND SAUCER" and "ROSES REGENCY TEACUP AND SAUCER" together highly suggests buying "PINK REGENCY TEACUP AND SAUCER" as a follow-up. This rule represents client preferences and the efficacy of product bundling in addition to indicating a high degree of correlation.

The strength and dependability of these correlations are revealed by the metrics used to assess these rules, including support, confidence, lift, leverage, and conviction. While confidence evaluates the possibility that the subsequent item will be purchased when the antecedent items are purchased, support measures the itemset's prevalence in the dataset. Leverage quantifies the dependence between the antecedent and consequent, conviction gives an assessment of the rule's reliability, and lift shows a rule's strength over the random co-occurrence of the antecedent and consequent.

  
Citing reliable sources is essential for validating and enhancing our understanding of association rule mining. Sources like "Mining association rules between sets of items in large databases" by Agrawal, R., Imieliński, T., & Swami, A. (1993); "Data Mining: Concepts and Techniques" by Han, J., Pei, J., & Kamber, M. (2011); and "The Elements of Statistical Learning" by Hastie, T., Tibshirani, R., & Friedman, J. (2009)" by Hastie, T., Tibshirani, R., & Friedman, J. (2009)" provide foundational knowledge and methodologies for thorough academic analysis. Additional research on the practical application of mining association rules in "arules – A Computational Environment for Mining Association Rules and Frequent Item Sets" (Hahsler, M., Grün, B., & Hornik, K., 2005) provides insights into the utilisation and application of these techniques in real-world datasets.  
  

<span style="color: var(--tw-prose-bold); font-weight: 600; font-family: -apple-system, BlinkMacSystemFont, sans-serif;">Bibliography</span>  

Agrawal, R., Imieliński, T., & Swami, A. (1993). Mining association rules between sets of items in large databases. In Proceedings of the 1993 ACM SIGMOD International Conference on Management of Data (pp. 207-216).

Han, J., Pei, J., & Kamber, M. (2011). Data Mining: Concepts and Techniques. Morgan Kaufmann.

Hastie, T., Tibshirani, R., & Friedman, J. (2009). The Elements of Statistical Learning. Springer Series in Statistics.

Hahsler, M., Grün, B., & Hornik, K. (2005). arules – A Computational Environment for Mining Association Rules and Frequent Item Sets. Journal of Statistical Software, 14(15).